<a href="https://colab.research.google.com/github/Chimatanagagopal/RAG/blob/main/HybridRAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [28]:
!pip install -q pypdf

In [29]:
from google.colab import files
from pypdf import PdfReader

uploaded = files.upload()
pdf_name = list(uploaded.keys())[0]

reader    = PdfReader(pdf_name)
full_text = ""

for page in reader.pages:
    text = page.extract_text()
    if text:
        full_text += text + "\n"

print(f"Pages      : {len(reader.pages)}")
print(f"Characters : {len(full_text):,}")
print(f"\n--- First 300 chars ---\n")
print(full_text[:300])

Saving PYTHON PROGRAMMING NOTES.pdf to PYTHON PROGRAMMING NOTES (3).pdf
Pages      : 142
Characters : 139,362

--- First 300 chars ---

PYTHON PROGRAMMING                                      III YEAR/II SEM                  MRCET 
 
 
PYTHON PROGRAMMING 
[R17A0554] 
LECTURE NOTES 
B.TECH III YEAR – II SEM (R17) 
(2019-20) 
 
 
 
 
DEPARTMENT OF 
COMPUTER SCIENCE AND ENGINEERING 
 
MALLA REDDY COLLEGE OF ENGINEERING & 
TECHNOLOGY 
(


In [30]:
CHUNK_SIZE = 1000
OVERLAP    = 200

chunks = []
start  = 0

while start < len(full_text):
    chunks.append(full_text[start : start + CHUNK_SIZE])
    start += CHUNK_SIZE - OVERLAP

print(f"Total chunks : {len(chunks)}")
print(f"Chunk size   : {CHUNK_SIZE} chars")
print(f"Overlap      : {OVERLAP} chars")

Total chunks : 175
Chunk size   : 1000 chars
Overlap      : 200 chars


In [31]:
i = 0
print(f"=== Chunk {i} ===")
print(f"Length: {len(chunks[i])} chars\n")
print(chunks[i])

=== Chunk 0 ===
Length: 1000 chars

PYTHON PROGRAMMING                                      III YEAR/II SEM                  MRCET 
 
 
PYTHON PROGRAMMING 
[R17A0554] 
LECTURE NOTES 
B.TECH III YEAR – II SEM (R17) 
(2019-20) 
 
 
 
 
DEPARTMENT OF 
COMPUTER SCIENCE AND ENGINEERING 
 
MALLA REDDY COLLEGE OF ENGINEERING & 
TECHNOLOGY 
(Autonomous Institution – UGC, Govt. of India) 
Recognized under 2(f) and 12 (B) of UGC ACT 1956 
(Affiliated to JNTUH, Hyderabad, Approved by AICTE - Accredited by NBA & NAAC – ‘A’ Grade - ISO 9001:2015 
Certified) 
Maisammaguda, Dhulapally (Post Via. Hakimpet), Secunderabad – 500100, Telangana State, India 
 
PYTHON PROGRAMMING                                      III YEAR/II SEM                  MRCET 
 
 
SYLLABUS 
MALLA REDDY COLLEGE OF ENGINEERING AND TECHNOLOGY 
 
III Year B. Tech CSE -II SEM       L T/P/D C 
             3  - / - / -  3 
OPEN ELECTIVE III 
(R17A0554) PYTHON PROGRAMMING 
OBJECTIVES:  
 To read and write simple Python programs. 
 To

In [20]:
!pip install -q sentence-transformers faiss-cpu

Load Embedding Model

In [21]:
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("all-MiniLM-L6-v2")
print("Model loaded!")
print("Embedding dimension:", embedder.get_sentence_embedding_dimension())

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Model loaded!
Embedding dimension: 384


/tmp/ipykernel_3930/3420059995.py:5: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("Embedding dimension:", embedder.get_sentence_embedding_dimension())


Embed All Chunks

In [22]:
import numpy as np

embeddings = embedder.encode(chunks, show_progress_bar=True)
embeddings = np.array(embeddings).astype("float32")

print("Shape:", embeddings.shape)
print("Meaning: ", embeddings.shape[0], "chunks →", embeddings.shape[1], "dimensions each")

Batches:   0%|          | 0/6 [00:00<?, ?it/s]

Shape: (175, 384)
Meaning:  175 chunks → 384 dimensions each


In [23]:
import faiss

dim         = embeddings.shape[1]
faiss_index = faiss.IndexFlatL2(dim)
faiss_index.add(embeddings)

print("Vectors stored:", faiss_index.ntotal)

Vectors stored: 175


In [24]:
query       = "What is this document about?"
query_embed = embedder.encode([query]).astype("float32")

D, I = faiss_index.search(query_embed, 5)

print("Top 5 matching chunks:\n")
for rank, (dist, idx) in enumerate(zip(D[0], I[0]), start=1):
    print(f"--- Rank {rank} | Distance: {dist:.4f} ---")
    print(chunks[idx][:300])
    print()

Top 5 matching chunks:

--- Rank 1 | Distance: 1.4726 ---
 1, 2: 4, 3: 9, 4: 16, 5: 25} 
  
PYTHON PROGRAMMING                                      III YEAR/II SEM                  MRCET 
103 
 
UNIT – V 
FILES, EXCEPTIONS, MODULES, PACKAGES 
Files and exception:  text files, reading and writing files, command line arguments, errors 
and exceptions, handli

--- Rank 2 | Distance: 1.5733 ---
plate 
This class is used to create a string template for simpler string substitutions 
Syntax: from string import Template 
Python arrays: 
Array is a container which can hold a fix number of items and these items should be of the 
same type. Most of the data structures make use of arrays to implem

--- Rank 3 | Distance: 1.5762 ---
n: 
Syntax: true_value if Condition else false_value 
>>> x = "1" if True else "2" 
>>> x 
'1' 
Statements: 
A statement is an instruction that the Python interpreter can execute. We have normally two 
basic statements , the assignment statement and the print statement.

In [25]:
!pip install -q rank_bm25

In [26]:
from rank_bm25 import BM25Okapi

tokenized_chunks = [chunk.lower().split() for chunk in chunks]
bm25             = BM25Okapi(tokenized_chunks)

print("BM25 index ready!")
print("Documents indexed:", len(tokenized_chunks))

BM25 index ready!
Documents indexed: 175


In [27]:
query       = "What is this document about?"
query_tokens = query.lower().split()

scores      = bm25.get_scores(query_tokens)
top_indices = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:5]

print("Top 5 BM25 results:\n")
for rank, idx in enumerate(top_indices, start=1):
    print(f"--- Rank {rank} | BM25 Score: {scores[idx]:.4f} ---")
    print(chunks[idx][:300])
    print()

Top 5 BM25 results:

--- Rank 1 | BM25 Score: 6.0280 ---
 r * r 
    return (c, a) 
print(circleInfo(10)) 
Output: 
C:/Users/MRCET/AppData/Local/Programs/Python/Python38-32/functupretval.py 
(62.8318, 314.159) 
PYTHON PROGRAMMING                                      III YEAR/II SEM                  MRCET 
96 
 
------------------------------ 
def f(x): 
 

--- Rank 2 | BM25 Score: 5.2119 ---
ython/Python38-32/pyyy/ex3.py 
number can not be divided by zero 
bye 
 The except block executes only when try block has an error, check it below  
a=5 
b=2 
try: 
    print(a/b) 
except Exception: 
    print("number can not be divided by zero") 
    print("bye") 
Output: 
C:/Users/MRCET/AppData/L

--- Rank 3 | BM25 Score: 5.1709 ---
 You are free to choose the 
number of spaces of indentation to use, but you then need to stick with it. 
Syntax errors: 
These are the most basic type of  error. They arise when the  Python parser   is unable to 
understand a line of code.  Syntax errors are almost 